In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('pima_diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.corr()['Outcome'] # how much our col. impact the Outcome

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [4]:
X = df.iloc[:,0:-1]
y = df.iloc[:,-1]

In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
X
X.shape

(768, 8)

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [7]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from keras.layers import Dense

In [8]:
model = Sequential()

model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy',metrics=['accuracy'])

C:\Users\shubham yadav\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6547 - loss: 0.6592 - val_accuracy: 0.6753 - val_loss: 0.6119
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6678 - loss: 0.6083 - val_accuracy: 0.7013 - val_loss: 0.5717
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7020 - loss: 0.5728 - val_accuracy: 0.7338 - val_loss: 0.5409
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7329 - loss: 0.5455 - val_accuracy: 0.7857 - val_loss: 0.5198
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7394 - loss: 0.5271 - val_accuracy: 0.7727 - val_loss: 0.5035
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7443 - loss: 0.5124 - val_accuracy: 0.7792 - val_loss: 0.4906
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7590 - loss: 0.5002 - val_accuracy: 0.7857 - val_loss: 0.4813
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7622 - loss: 0.4910 - val_accuracy: 0.7922 - 

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 965 (3.77 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 644 (2.52 KB)

Now we apply hyperperameter tunning 

In [11]:
# 1. we find the best optimizer
# 2. No. of nodes in a layer 
# 3. how to select no. of layer 
# 4. All in one model  

In [12]:
# we made this fun to find best optimizer 
import keras_tuner as kt 

def build_model(hp):
    model = Sequential()
    
    model.add(Dense(32, activation= 'relu', input_dim= 8))
    model.add(Dense(1,activation='sigmoid'))
    
    optimizer = hp.Choice('optimizer', values = ['adam','sgd','rmsprop','adadelta'])
   
    model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics=['accuracy'])
    return model 

In [13]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5)

In [14]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 4 Complete [00h 00m 01s]
val_accuracy: 0.5064935088157654

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 07s


In [15]:
tuner.get_best_hyperparameters()[0].values # this give the first(best) optimizer

{'optimizer': 'adam'}

In [16]:
# we need not to make model again we can get the model with best perameter by tuner 
model = tuner.get_best_models(num_models=1)[0] # we abestrect the top optimizer
model.summary()

C:\Users\shubham yadav\AppData\Roaming\Python\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.fit(X_train, y_train, epochs=100, batch_size=32, initial_epoch=6, validation_data=(X_test, y_test))
# now this model start traning with epoch 6 and go to till 100 bcoz we already done 5 epoch before.

# so this is the code to find the best optimizer 

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7622 - loss: 0.5170 - val_accuracy: 0.7662 - val_loss: 0.5014
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7655 - loss: 0.5022 - val_accuracy: 0.7792 - val_loss: 0.4911
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7720 - loss: 0.4918 - val_accuracy: 0.7727 - val_loss: 0.4839
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7704 - loss: 0.4846 - val_accuracy: 0.7857 - val_loss: 0.4780
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7704 - loss: 0.4782 - val_accuracy: 0.8052 - val_loss: 0.4731
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7736 - loss: 0.4735 - val_accuracy: 0.8052 - val_loss: 0.4706
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7785 - loss: 0.4695 - val_accuracy: 0.8117 - val_loss: 0.4681
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7769 - loss: 0.4662 - val_accuracy: 0.805

Now we find the NO. of nueron

In [18]:
def build_model(hp):
    model = Sequential()
    
    unit = hp.Int('unit', min_value =8, max_value =128, step=3)
    
    model.add(Dense(units=unit, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [19]:
tuner2 = kt.RandomSearch(
    build_model,
    objective= 'val_accuracy',
    max_trials=5,
    overwrite=True
)

In [20]:
tuner2.search(X_train, y_train, epochs =5 , validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7597402334213257

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 08s


In [21]:
tuner2.get_best_hyperparameters()[0]

In [22]:
# model = tuner.get_best_models(num_models=1)[0].values # we get 1st model (best)
# model.fit(X_train, y_train,batch_size=32, validation_data=(X_test, y_test))

Now we select the No. of layer

In [23]:
def build_model(hp):

    model = Sequential()
    model.add(Dense(72, activation='relu', input_dim=8))

    for i in range(hp.Int('num_layer', min_value = 1, max_value = 10)):
        model.add(Dense(72, activation='relu'))
        
    model.add(Dense(1,activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [24]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    overwrite = True
)

In [25]:
tuner.search(X_train, y_train, epochs = 5, validation_data=(X_test, y_test) )

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 15s


In [26]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 2}

In [27]:
model = tuner.get_best_models(num_models=1)[0]

model.fit(X_train , y_train , epochs=100, initial_epoch=6, validation_data = (X_test, y_test))

Epoch 7/100


C:\Users\shubham yadav\AppData\Roaming\Python\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7769 - loss: 0.4592 - val_accuracy: 0.8247 - val_loss: 0.4621
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7785 - loss: 0.4464 - val_accuracy: 0.8182 - val_loss: 0.4644
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7915 - loss: 0.4402 - val_accuracy: 0.8312 - val_loss: 0.4610
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7866 - loss: 0.4365 - val_accuracy: 0.8117 - val_loss: 0.4612
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7850 - loss: 0.4283 - val_accuracy: 0.8182 - val_loss: 0.4621
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7932 - loss: 0.4229 - val_accuracy: 0.8117 - val_loss: 0.4599
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7883 - loss: 0.4205 - val_accuracy: 0.8182 - val_loss: 0.4611
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8078 - loss: 0.4139 - val_accuracy: 0.8182 - val_los

Now we find all in one place 

In [ ]:
def build_model(counter,hp):                          
    model = Sequential()
    
    for i in range(hp.Int('num_layer', min_value = 1, max_value = 10)):
        if counter == 0:
        
            model.add(Dense(
                    hp.Int('unit'+ str[i], min_value=8, max_value=128, step=8),
                    activation= hp.choice('activation' + str[i], values=['relu','tanh','sigmoid'],
                    input_dim=8
                                          )
                    
            else:
                model.add(Dense(
                hp.Int('unit'+ str[i], min_value=8, max_value=128, step=8),
                activation= hp.choice('activation' + str[i], values=['relu','tanh','sigmoid'],
            ))
            counter+=1
                          
                          
                          

SyntaxError: invalid syntax (3758968396.py, line 13)